In [1]:
import collections, glob, json, math, os, time
import torch, torch.nn.functional as F, torchvision, timm
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
import schedulefree
from schedulefree import ScheduleFreeWrapper
from splus import SPlus

In [2]:
SMOKE = True        # True - быстрая проверка (крошечная модель, мало данных); False - полный эксперимент
METHODS_TO_RUN = ['adamw_cos', 'adamw_sf', 'splus', 'splus_noema_sf', 'splus_sf']
SEEDS = [0]

OUT = 'custom_experiment_results' + ('/smoke' if SMOKE else '')
EPOCHS        = 2  if SMOKE else 200
BS            = 64 if SMOKE else 128
WARMUP_EPOCHS = 1  if SMOKE else 5
INVERSE_EVERY = 10 if SMOKE else 50     # как часто пересчитывать собственные базисы SPlus
WORKERS       = 0  if SMOKE else 8       # процессы для аугментаций на CPU
LOG_EVERY     = 10 if SMOKE else 50      # как часто писать лосс по шагам в tensorboard

LR = {  # <-- подставьте lr из прошлого эксперимента
    'adamw_cos': 5e-4, 'adamw_sf': 5e-4,
    'splus': 0.2, 'splus_noema_sf': 0.2, 'splus_sf': 0.2,
}
# sched: 'cos' | 'const' (после warmup). evals: где оценивать на тесте ([] = живые веса, первая точка основная):
#   'sf' - усреднённая точка Schedule-Free, 'ema' - EMA-веса SPlus
METHODS = {
    'adamw_cos':      dict(wd=0.05, sched='cos',   evals=[]),
    'adamw_sf':       dict(wd=0.05, sched='const', evals=['sf']),
    'splus':          dict(wd=0.1,  sched='const', evals=['ema']),
    'splus_noema_sf': dict(wd=0.1,  sched='const', evals=['sf']),
    'splus_sf':       dict(wd=0.1,  sched='const', evals=['sf', 'ema']),  # SF + EMA: логируем обе точки
}
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(OUT, exist_ok=True)
print('device:', dev, '| results ->', OUT)

device: cpu | results -> custom_experiment_results/smoke


In [3]:
def make_data(seed):
    train_tf = T.Compose([T.RandomCrop(32, padding=4, padding_mode='reflect'), T.RandomHorizontalFlip(),
                          T.AutoAugment(T.AutoAugmentPolicy.CIFAR10), T.ToTensor(), T.Normalize(MEAN, STD)])
    test_tf = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])
    train = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=train_tf)
    test = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=test_tf)
    if SMOKE:
        train, test = Subset(train, range(2048)), Subset(test, range(512))
    kw = dict(num_workers=WORKERS, pin_memory=dev.type == 'cuda', persistent_workers=WORKERS > 0)
    train_loader = DataLoader(train, batch_size=BS, shuffle=True, drop_last=True,
                              generator=torch.Generator().manual_seed(seed), **kw)
    return train_loader, DataLoader(test, batch_size=500, **kw)


def make_model():
    if SMOKE:   # крошечная модель
        kw = dict(patch_size=8, embed_dim=64, depth=2, num_heads=4, mlp_ratio=2.0)
    else:       # рецепт omihub777/ViT-CIFAR, ~6.3M параметров
        kw = dict(patch_size=4, embed_dim=384, depth=7, num_heads=12, mlp_ratio=1.0)
    return timm.models.vision_transformer.VisionTransformer(img_size=32, num_classes=10, **kw)

print(f'params: {sum(p.numel() for p in make_model().parameters()) / 1e6:.2f}M')

params: 0.08M


In [4]:
class SFWrapper(ScheduleFreeWrapper):
    # ScheduleFreeWrapper кладёт свой буфер 'z' в base.state[p], а SPlus считает состояние неинициализированным
    # только пока len(state) == 0. Чтобы они не мешали друг другу, храним состояние обёртки отдельно.
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self._sf_state = collections.defaultdict(dict)

    @property
    def state(self):
        return self._sf_state


def build(name, model, lr):
    m = METHODS[name]
    if name == 'adamw_cos':
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=m['wd'])
    elif name == 'adamw_sf':
        opt = schedulefree.AdamWScheduleFree(model.parameters(), lr=lr, weight_decay=m['wd'])
    else:
        head = list(model.head.parameters())
        ids = {id(p) for p in head}
        body = [p for p in model.parameters() if id(p) not in ids]
        sf = name.endswith('_sf')
        # голова: max_dim=0 -> без собственного базиса, обновление sign(momentum) (как в статье)
        base = SPlus([{'params': body}, {'params': head, 'max_dim': 0}], lr=lr,
                     ema_rate=None if name == 'splus_noema_sf' else 0.999,
                     weight_decay=m['wd'], inverse_every=INVERSE_EVERY)
        opt = SFWrapper(base, momentum=0.9) if sf else base
    for g in opt.param_groups:
        g['base_lr'] = g['lr']
    if 'sf' in m['evals']:
        opt.train()                      
    return opt


def lr_factor(step, total, warm, sched, min_frac=0.01):
    w = min(1.0, (step + 1) / max(1, warm))
    if sched == 'const':
        return w
    p = min(1.0, step / max(1, total))
    return w * (min_frac + (1 - min_frac) * 0.5 * (1 + math.cos(math.pi * p)))

In [5]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss = correct = n = 0
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        with torch.autocast(dev.type, torch.bfloat16, enabled=dev.type == 'cuda'):
            out = model(x)
        loss += F.cross_entropy(out.float(), y, reduction='sum').item()
        correct += (out.argmax(1) == y).sum().item()
        n += len(y)
    return loss / n, correct / n


def eval_at(opt, point, model, loader):
    if point is None:                                
        return evaluate(model, loader)
    o = opt if point == 'sf' else getattr(opt, 'base', opt)    # 'ema' -> сам SPlus (внутри обёртки, если она есть)
    o.eval()
    try:
        return evaluate(model, loader)
    finally:
        o.train()

In [6]:
def run(name, seed):
    lr, m = LR[name], METHODS[name]
    tag = f'{name}_lr{lr:.4g}_s{seed}'
    path = f'{OUT}/{tag}.json'
    if os.path.exists(path):
        print('skip', tag)
        return
    amp = dev.type == 'cuda'
    torch.manual_seed(seed)     
    model = make_model().to(dev)
    opt = build(name, model, lr)
    train_loader, test_loader = make_data(seed)
    spe = len(train_loader)
    total, warm = EPOCHS * spe, int(WARMUP_EPOCHS * spe)
    points = m['evals'] or [None]

    log = dict(cfg=dict(method=name, lr=lr, seed=seed, epochs=EPOCHS, bs=BS, inverse_every=INVERSE_EVERY, **m),
               train_loss=[], train_acc=[], test_loss=[], test_acc=[])
    if len(points) > 1:
        log.update(test_loss_ema=[], test_acc_ema=[])
    writer = SummaryWriter(f'{OUT}/tb/{tag}')
    step, t0, diverged = 0, time.time(), False

    for ep in range(EPOCHS):
        model.train()
        sum_loss, sum_ok, n = torch.zeros((), device=dev), torch.zeros((), device=dev), 0
        for x, y in train_loader:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            f = lr_factor(step, total, warm, m['sched'])
            for g in opt.param_groups:
                g['lr'] = g['base_lr'] * f
            opt.zero_grad(set_to_none=True)
            with torch.autocast(dev.type, torch.bfloat16, enabled=amp):
                out = model(x)
            loss = F.cross_entropy(out.float(), y, label_smoothing=0.1)
            loss.backward()
            opt.step()
            sum_loss += loss.detach()
            sum_ok += (out.argmax(1) == y).sum()
            n += len(y)
            step += 1
            if step % LOG_EVERY == 0:                    # .item() синхронизирует GPU, поэтому не на каждом шаге
                v = loss.item()
                writer.add_scalar('step/train_loss', v, step)
                writer.add_scalar('step/lr_factor', f, step)
                if not math.isfinite(v):
                    diverged = True
                    break
        if diverged:
            print(f'{tag}: DIVERGED at step {step}')
            break

        res = [eval_at(opt, p, model, test_loader) for p in points]
        tr_loss, tr_acc = sum_loss.item() / (n / BS), sum_ok.item() / n
        log['train_loss'].append(tr_loss); log['train_acc'].append(tr_acc)
        log['test_loss'].append(res[0][0]); log['test_acc'].append(res[0][1])
        writer.add_scalar('epoch/train_loss', tr_loss, ep); writer.add_scalar('epoch/train_acc', tr_acc, ep)
        writer.add_scalar('epoch/test_loss', res[0][0], ep); writer.add_scalar('epoch/test_acc', res[0][1], ep)
        if len(res) > 1:
            log['test_loss_ema'].append(res[1][0]); log['test_acc_ema'].append(res[1][1])
            writer.add_scalar('epoch/test_acc_ema', res[1][1], ep)
        if (ep + 1) % (1 if SMOKE else 5) == 0 or ep == EPOCHS - 1:
            extra = f' | ema acc {res[1][1]:.4f}' if len(res) > 1 else ''
            print(f'{tag} ep {ep + 1}/{EPOCHS} train {tr_loss:.3f} test {res[0][0]:.3f} '
                  f'acc {res[0][1]:.4f}{extra} | {time.time() - t0:.0f}s', flush=True)

    log['time'], log['diverged'] = time.time() - t0, diverged
    writer.close()
    json.dump(log, open(path, 'w'))
    del model, opt, train_loader, test_loader
    torch.cuda.empty_cache()

In [8]:
for seed in SEEDS:                    
    for name in METHODS_TO_RUN:
        run(name, seed)

/opt/homebrew/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


adamw_cos_lr0.0005_s0 ep 1/2 train 2.269 test 2.135 acc 0.2363 | 1s
adamw_cos_lr0.0005_s0 ep 2/2 train 2.205 test 2.104 acc 0.2539 | 2s
adamw_sf_lr0.0005_s0 ep 1/2 train 2.272 test 2.137 acc 0.2168 | 1s
adamw_sf_lr0.0005_s0 ep 2/2 train 2.197 test 2.087 acc 0.2402 | 2s
skip splus_lr0.2_s0
skip splus_noema_sf_lr0.2_s0
skip splus_sf_lr0.2_s0


In [9]:
for f in sorted(glob.glob(f'{OUT}/*.json')):
    r = json.load(open(f))
    acc = f"{r['test_acc'][-1]:.4f}" if r['test_acc'] else '-'
    ema = f" | ema acc {r['test_acc_ema'][-1]:.4f}" if r.get('test_acc_ema') else ''
    print(f"{r['cfg']['method']:15s} seed {r['cfg']['seed']}  lr {r['cfg']['lr']:<8g} final test acc {acc}{ema}"
          f"{'  DIVERGED' if r['diverged'] else ''}  ({r['time']:.0f}s)")

adamw_cos       seed 0  lr 0.0005   final test acc 0.2539  (2s)
adamw_cos       seed 0  lr 0.001    final test acc 0.2559  (2s)
adamw_sf        seed 0  lr 0.0005   final test acc 0.2402  (2s)
adamw_sf        seed 0  lr 0.001    final test acc 0.2344  (2s)
splus           seed 0  lr 0.2      final test acc 0.2773  (2s)
splus_noema_sf  seed 0  lr 0.2      final test acc 0.2324  (2s)
splus_sf        seed 0  lr 0.2      final test acc 0.2441 | ema acc 0.2754  (2s)
